# pandas for Machine Learning
### A junior ML engineer's path to confidence with pandas

pandas is the layer where raw, messy, labeled data becomes a clean numeric table you can hand to a model. NumPy gave you fast arrays of one type; pandas wraps a table of NumPy columns with **labels** (row and column names) and the tools to clean, select, group, and reshape it.

## Table of contents
0. [Introduction](#0.-Introduction)
1. [Series and DataFrames: creating and loading](#1.-Series-and-DataFrames:-creating-and-loading)
2. [Inspecting a DataFrame](#2.-Inspecting-a-DataFrame)
3. [Selecting data: loc, iloc, and boolean masks](#3.-Selecting-data:-loc,-iloc,-and-boolean-masks)
4. [Adding, transforming, and dropping columns](#4.-Adding,-transforming,-and-dropping-columns)
5. [Sorting, counting, and summarizing](#5.-Sorting,-counting,-and-summarizing)
6. [Handling missing data](#6.-Handling-missing-data)
7. [groupby: split-apply-combine](#7.-groupby:-split-apply-combine)
8. [Combining DataFrames: concat and merge](#8.-Combining-DataFrames:-concat-and-merge)
9. [Reshaping: pivot and melt](#9.-Reshaping:-pivot-and-melt)
10. [From pandas to the ML stack](#10.-From-pandas-to-the-ML-stack)
11. [Summary](#11.-Summary)
12. [Exercises](#12.-Exercises)

## 0. Introduction

**pandas** is the library you reach for the moment your data has *columns with names*: a CSV of house prices, a table of users, the output of a SQL query. NumPy handles a block of numbers of one type; pandas handles a real-world **table** where the "age" column is an integer, the "city" column is text, and the "price" column has a few missing values, all lined up by a shared row index.

Every ML workflow starts here. Before scikit-learn ever sees your data, pandas is what loads it, cleans it, engineers features from it, and finally hands over a clean numeric array.

**Convention:** pandas is always imported as `pd`. You'll see this in essentially every ML codebase, so stick to it.

In [1]:
import pandas as pd

print("pandas version:", pd.__version__)

pandas version: 3.0.3


### 0.1 A quick taste: the same job, two ways

Say you have some game scores tagged by team, and you want the **average score per team**. In plain Python you reach for a loop and a couple of dictionaries to accumulate the totals:

In [2]:
teams  = ["A", "A", "B", "B", "B"]
scores = [10, 20, 30, 40, 50]

sums, counts = {}, {}
for team, score in zip(teams, scores):
    sums[team] = sums.get(team, 0) + score
    counts[team] = counts.get(team, 0) + 1

averages = {team: sums[team] / counts[team] for team in sums}
print(averages)

{'A': 15.0, 'B': 40.0}


That works, but it is all bookkeeping: two dictionaries, a manual loop, a final comprehension. With pandas the same idea is one line that reads like the sentence "group by team, take the mean of score":

In [3]:
df = pd.DataFrame({"team": teams, "score": scores})
df.groupby("team")["score"].mean()

team
A    15.0
B    40.0
Name: score, dtype: float64

That `groupby(...).mean()` pattern is the spine of the whole library, and Chapter 7 builds up to it deliberately. Everything before it is about getting your data *into* a clean table you can group.

### 0.2 Why this is possible: labeled columns of arrays

A `DataFrame` is a dictionary of **columns**, where each column is a NumPy array of one type, and all columns share one **index** (the row labels). Because each column is a real array, whole-column math is vectorized and fast (the NumPy lesson carries straight over). Because the columns and rows carry **labels**, you select by *name* (`df["score"]`) instead of by remembering which position a column lives in. That labeling is what makes pandas code readable months later.

### 0.3 A first taste of the payoff: speed

The one-liner is not just shorter, it is far faster on real data, because the grouping runs in compiled code instead of a Python loop. Here is a quick, honest preview on half a million rows; the full benchmark comes in Chapter 7.

In [4]:
import numpy as np
import time

rng = np.random.default_rng(0)
n = 500_000
preview = pd.DataFrame({"team": rng.integers(0, 100, n), "score": rng.random(n)})

start = time.perf_counter()
py = {}
for k, v in zip(preview["team"].tolist(), preview["score"].tolist()):
    py.setdefault(k, []).append(v)
py_means = {k: sum(vs) / len(vs) for k, vs in py.items()}
loop_time = time.perf_counter() - start

start = time.perf_counter()
pd_means = preview.groupby("team")["score"].mean()
pandas_time = time.perf_counter() - start

print(f"pure Python loop: {loop_time:.3f}s")
print(f"pandas groupby:   {pandas_time:.4f}s")
print(f"speedup:          {loop_time / pandas_time:.0f}x")

pure Python loop: 0.042s
pandas groupby:   0.0047s
speedup:          9x


Same answer, a fraction of the time, and much less code to get wrong. Let's build the mental model that gets us there, one piece at a time.

## 1. Series and DataFrames: creating and loading

Two objects, and you cannot inspect or select anything until you can make them. A **`Series`** is a single labeled column (a 1-D array plus an index). A **`DataFrame`** is a table: several `Series` sharing one index. This is where you start, because everything else needs an object in hand.

### 1.1 `Series`: one labeled column

A `Series` is the building block. It is a 1-D array of values plus an **index** of labels, one per value. If you don't supply an index, you get the default integer positions `0, 1, 2, ...`.

In [5]:
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s)
print()
print("value at label 'b':", s["b"])
print("underlying array:", s.to_numpy())

a    10
b    20
c    30
dtype: int64

value at label 'b': 20
underlying array: [10 20 30]


### 1.2 `DataFrame`: a table from a dict of columns

The most common way to build one by hand is a **dict of columns**: each key is a column name, each value is the list of that column's entries. Every column shares the same row index.

In [6]:
people = pd.DataFrame({
    "name": ["ann", "bob", "cara"],
    "age":  [25, 30, 35],
    "city": ["NYC", "LA", "SF"],
})
people

,name,age,city
0,ann,25,NYC
1,bob,30,LA
2,cara,35,SF


### 1.3 Pure Python vs pandas: holding a table

In plain Python you'd hold the same table as a dict of lists, and answering even a simple question ("the average age") means writing a loop. The DataFrame gives you the answer as a method call, and prints as a real table instead of raw brackets.

In [7]:
# plain Python: a dict of lists, and a manual average
raw = {"name": ["ann", "bob", "cara"], "age": [25, 30, 35]}
avg_age_py = sum(raw["age"]) / len(raw["age"])
print("dict of lists:", raw)
print("average age (loop):", avg_age_py)

# pandas: the column is an array, so the aggregate is a method
print("average age (pandas):", people["age"].mean())

dict of lists: {'name': ['ann', 'bob', 'cara'], 'age': [25, 30, 35]}
average age (loop): 30.0
average age (pandas): 30.0


### 1.4 Loading real data

You will rarely type data in by hand. In real work it comes from a file (`pd.read_csv("data.csv")` is the one you'll use most) or from a library. Throughout this notebook, you'll work with the classic **Iris** dataset, which scikit-learn hands you directly as a DataFrame with a single call: 150 flowers, four measured features, and the species each one belongs to.

In [8]:
from sklearn.datasets import load_iris

bunch = load_iris(as_frame=True)
iris = bunch.frame.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)":  "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)":  "petal_width",
})
# turn the numeric target (0/1/2) into a readable species column
iris["species"] = iris["target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})
iris = iris.drop(columns="target")
iris.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


> **ML tie-in:** loading data into a DataFrame is step one of every project. `read_csv` for files, a single-call loader for the standard datasets, and (later) a SQL query all land you in the same object, so everything you learn next applies no matter where the data came from.

## 2. Inspecting a DataFrame

You just loaded 150 rows you have never seen. Before any modeling, you look: how big is it, what are the columns, what type is each one, and what do the numbers roughly look like. These few methods are the first thing you run on *any* new dataset.

### 2.1 The anatomy: index, columns, values

Every DataFrame is three parts: the **columns** (labels across the top), the **index** (labels down the side), and the **values** (the grid itself). Keep this picture in mind; every selection in Chapter 3 is just "pick some columns and some index labels."

![DataFrame anatomy: index, columns, values](assets/pandas/diagram_anatomy.png)

In [9]:
small = pd.DataFrame(
    {"name": ["ann", "bob", "cara"], "age": [25, 30, 35], "city": ["NYC", "LA", "SF"]}
)
print("shape (rows, cols):", small.shape)
print("columns:", list(small.columns))
print("index:  ", list(small.index))
print("values (a NumPy array):\n", small.values)

shape (rows, cols): (3, 3)
columns: ['name', 'age', 'city']
index:   [0, 1, 2]
values (a NumPy array):
 [['ann' 25 'NYC']
 ['bob' 30 'LA']
 ['cara' 35 'SF']]


### 2.2 `.dtypes`: one type per column

Unlike a single NumPy array (one type for everything), a DataFrame carries **one dtype per column**: numbers stay numeric, text is stored as `object` (or the newer `string` type). This is the whole point of pandas: mixed-type tables.

In [10]:
iris.dtypes

sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
species             str
dtype: object

### 2.3 `.head()` and `.tail()`: peek at the ends

You almost never print a whole table. `.head(n)` shows the first `n` rows (default 5), `.tail(n)` the last.

In [11]:
iris.head(3)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa


### 2.4 `.info()`: the structural summary

`.info()` is the one-shot overview: how many rows, every column with its non-null count and dtype, and the memory footprint. The non-null counts are how you spot missing data at a glance (more in Chapter 6).

In [12]:
iris.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


### 2.5 `.describe()`: quick statistics per numeric column

`.describe()` gives count, mean, std, min, quartiles, and max for every numeric column: a fast sense of scale and spread before you model anything.

In [13]:
iris.describe()

,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


> **ML tie-in:** `shape`, `info`, and `describe` are the reflexive first three commands on any new dataset. They tell you how much data you have, which columns are numeric vs categorical, and whether features are on wildly different scales (a hint you'll need to standardize later).

## 3. Selecting data: loc, iloc, and boolean masks

Selecting the right rows and columns is the single most-used skill in pandas, and label-based selection is half of the library's spine. There are three tools: pick columns by name, pick by **label** with `.loc`, pick by **position** with `.iloc`, and filter rows with a **boolean mask**.

### 3.1 Selecting columns by name

Indexing with a single column name returns a `Series`; indexing with a **list** of names returns a `DataFrame` (note the double brackets). This distinction trips up everyone once.

In [14]:
print(type(iris["petal_length"]))       # one name -> Series
print(type(iris[["petal_length", "species"]]))  # list of names -> DataFrame
iris[["petal_length", "species"]].head(3)

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,petal_length,species
0,1.4,setosa
1,1.4,setosa
2,1.3,setosa


### 3.2 `.loc` vs `.iloc`: label vs position

This is the distinction to burn in. `.loc` selects by the **label** in the index; `.iloc` selects by **integer position**, exactly like a Python list. They differ whenever the index isn't a plain `0, 1, 2, ...` range.

![loc selects by label, iloc selects by position](assets/pandas/diagram_loc_iloc.png)

In [15]:
lbl = pd.DataFrame({"x": [10, 20, 30, 40], "y": [1, 2, 3, 4]}, index=["a", "b", "c", "d"])
print("loc['b']  (row labeled 'b'):")
print(lbl.loc["b"])
print()
print("iloc[2]   (row at position 2):")
print(lbl.iloc[2])

loc['b']  (row labeled 'b'):
x    20
y     2
Name: b, dtype: int64

iloc[2]   (row at position 2):
x    30
y     3
Name: c, dtype: int64


### 3.3 `.loc` and `.iloc` for rows and columns together

Both take a row selector and a column selector, separated by a comma: `df.loc[rows, cols]`. With `.loc`, a label range like `"a":"c"` is **inclusive** of the end (unlike Python slicing). This is how you grab a sub-block of the table.

In [16]:
# rows 0..2, only the two petal columns, by label
print(iris.loc[0:2, ["petal_length", "petal_width"]])
print()
# first 3 rows, first 2 columns, by position
print(iris.iloc[0:3, 0:2])

   petal_length  petal_width
0           1.4          0.2
1           1.4          0.2
2           1.3          0.2

   sepal_length  sepal_width
0           5.1          3.5
1           4.9          3.0
2           4.7          3.2


### 3.4 Boolean masks: filtering rows by a condition

A comparison on a column produces a boolean `Series`; indexing the DataFrame with it keeps only the `True` rows. Combine conditions with `&` (and) and `|` (or), each parenthesized. This is how you filter data every single day.

In [17]:
big_petals = iris[iris["petal_length"] > 5]
print("rows with petal_length > 5:", big_petals.shape[0])

# combine conditions: virginica with narrow petals
mask = (iris["species"] == "virginica") & (iris["petal_width"] < 2.0)
iris[mask].head(3)

rows with petal_length > 5: 42


,sepal_length,sepal_width,petal_length,petal_width,species
101,5.8,2.7,5.1,1.9,virginica
103,6.3,2.9,5.6,1.8,virginica
106,4.9,2.5,4.5,1.7,virginica


### 3.5 Pure Python vs pandas: filtering

The plain-Python way to filter is a loop or comprehension over rows, rebuilding the result yourself. The boolean mask *is* the filter: one readable expression, no loop, and it runs in compiled code.

In [18]:
# plain Python: comprehension over paired lists
lengths = iris["petal_length"].tolist()
species = iris["species"].tolist()
py_filtered = [sp for length, sp in zip(lengths, species) if length > 5]
print("loop count:  ", len(py_filtered))

# pandas: the mask does it
print("pandas count:", len(iris[iris["petal_length"] > 5]))

loop count:   42
pandas count: 42


> **ML tie-in:** this is exactly how you carve a dataset into `X` and `y`. Feature columns are a list-of-names selection (`iris[feature_cols]`); the target is a single-column selection (`iris["species"]`); dropping outliers or picking a subgroup is a boolean mask.

## 4. Adding, transforming, and dropping columns

Real datasets rarely arrive with the columns you want. Feature engineering is mostly this: compute a new column from existing ones, transform a column, and drop what you don't need.

### 4.1 A new column from a vectorized computation

Assigning to a new column name creates it. Because columns are arrays, the arithmetic is vectorized: no loop, one value per row. Here's a crude "petal area" feature, built the same way.

In [19]:
iris["petal_area"] = iris["petal_length"] * iris["petal_width"]
iris[["petal_length", "petal_width", "petal_area"]].head(3)

,petal_length,petal_width,petal_area
0,1.4,0.2,0.28
1,1.4,0.2,0.28
2,1.3,0.2,0.26


### 4.2 `.assign`: add columns without mutating

`df["new"] = ...` changes the DataFrame in place. `.assign(new=...)` instead returns a **new** DataFrame with the column added, which is what you want inside a chain of operations that shouldn't touch the original.

In [20]:
with_ratio = iris.assign(petal_ratio=iris["petal_length"] / iris["petal_width"])
print("original still has", iris.shape[1], "columns")
print("the copy has", with_ratio.shape[1], "columns")
with_ratio[["petal_length", "petal_width", "petal_ratio"]].head(3)

original still has 6 columns
the copy has 7 columns


,petal_length,petal_width,petal_ratio
0,1.4,0.2,7.0
1,1.4,0.2,7.0
2,1.3,0.2,6.5


### 4.3 `.map`: transform a column value by value

`.map` applies a lookup or a function to every entry of a `Series`. It's the clean way to recode a categorical column, here turning species names into short codes.

In [21]:
codes = iris["species"].map({"setosa": "se", "versicolor": "ve", "virginica": "vi"})
codes.head(3)

0    se
1    se
2    se
Name: species, dtype: str

### 4.4 Dropping columns and rows with `.drop`

`.drop(columns=[...])` removes columns; `.drop(index=[...])` removes rows by label. It returns a new DataFrame, so assign the result if you want to keep it.

In [22]:
trimmed = iris.drop(columns=["petal_area"])
print("before:", list(iris.columns))
print("after: ", list(trimmed.columns))

before: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species', 'petal_area']
after:  ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']


> **ML tie-in:** every engineered feature (a ratio, a log, an interaction term, a recoded category) is one of these operations. Building good columns is often where most of a model's accuracy actually comes from.

## 5. Sorting, counting, and summarizing

A handful of everyday verbs for understanding a column: order the rows, count categories, and reduce a column to a single number. These also warm up the "summarize per group" idea that Chapter 7 generalizes.

### 5.1 `.sort_values`: order the rows

Sort by one column, or by several (the later columns break ties). `ascending=False` flips the order. It returns a new, reordered DataFrame.

In [23]:
iris.sort_values("petal_length", ascending=False).head(3)

,sepal_length,sepal_width,petal_length,petal_width,species,petal_area
118,7.7,2.6,6.9,2.3,virginica,15.87
117,7.7,3.8,6.7,2.2,virginica,14.74
122,7.7,2.8,6.7,2.0,virginica,13.40


### 5.2 `.value_counts`: how many of each category

`.value_counts()` on a column tallies how often each distinct value appears, sorted most-common first. Pass `normalize=True` for proportions instead of raw counts.

In [24]:
print(iris["species"].value_counts())
print()
print(iris["species"].value_counts(normalize=True))

species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

species
setosa        0.333333
versicolor    0.333333
virginica     0.333333
Name: proportion, dtype: float64


### 5.3 `.unique` and `.nunique`: the distinct values

`.unique()` lists the distinct values; `.nunique()` just counts them. Quick way to see the categories in a column, or to confirm an ID column really is unique.

In [25]:
print("distinct species:", iris["species"].unique())
print("how many:", iris["species"].nunique())

distinct species: <StringArray>
['setosa', 'versicolor', 'virginica']
Length: 3, dtype: str
how many: 3


### 5.4 Column-wide aggregates

A whole column reduces to one number with `.mean()`, `.sum()`, `.min()`, `.max()`, `.std()`, just like NumPy. In Chapter 7, you'll compute these *per group* instead of over the whole column.

In [26]:
print("mean petal_length:", round(iris["petal_length"].mean(), 3))
print("max  petal_length:", iris["petal_length"].max())

mean petal_length: 3.758
max  petal_length: 6.9


> **ML tie-in:** `value_counts` on the target column is how you check **class balance** before training a classifier; a heavily skewed target changes how you evaluate the model. Sorting surfaces the extreme rows worth inspecting.

## 6. Handling missing data

Real datasets have holes: a sensor dropped out, a field was left blank. pandas marks missing entries as `NaN`, and models cannot train on `NaN`. Finding and dealing with it is unavoidable preprocessing. Iris is clean, so here's a small table with gaps to work on instead.

In [27]:
data = pd.DataFrame({
    "height": [150, 160, None, 175, 168],
    "weight": [50, None, 65, 80, None],
    "city":   ["NYC", "LA", "SF", None, "LA"],
})
data

,height,weight,city
0,150.0,50.0,NYC
1,160.0,NaN,LA
2,NaN,65.0,SF
3,175.0,80.0,NaN
4,168.0,NaN,LA


### 6.1 Finding missing values

`.isna()` returns a same-shaped boolean table (`True` where a value is missing). Summing it per column, `.isna().sum()`, gives the count of holes in each column: the first thing to check.

In [28]:
data.isna().sum()

height    1
weight    2
city      1
dtype: int64

### 6.2 `.dropna`: remove rows with holes

`.dropna()` drops every row containing any `NaN`. Simple, but it can throw away a lot of data, so it's the right call only when missing rows are few.

In [29]:
print("before:", data.shape)
print("after dropna:", data.dropna().shape)
data.dropna()

before: (5, 3)
after dropna: (1, 3)


,height,weight,city
0,150.0,50.0,NYC


### 6.3 `.fillna`: fill the holes instead

Often you'd rather keep the rows and fill the gaps. `.fillna(value)` substitutes a constant; more usefully, fill a numeric column with its own mean (a basic **imputation**). Here, `weight` gets filled with the mean weight.

In [30]:
filled = data.copy()
filled["weight"] = filled["weight"].fillna(filled["weight"].mean())
filled

,height,weight,city
0,150.0,50.0,NYC
1,160.0,65.0,LA
2,NaN,65.0,SF
3,175.0,80.0,NaN
4,168.0,65.0,LA


> **ML tie-in:** deciding what to do with missing values is a real modeling choice. Mean/median imputation is the common baseline (scikit-learn's `SimpleImputer` automates exactly this); `.isna().sum()` is how you audit the damage before choosing.

## 7. groupby: split-apply-combine

This is the heart of pandas. Almost every "per category" question, average price per neighborhood, class balance per label, total sales per region, is the same three-step move: **split** the rows into groups, **apply** an aggregate to each group, **combine** the results into one table. You'll build it the hard way first, then let `groupby` collapse it.

### 7.1 The manual way (on purpose)

Without `groupby`, computing a per-group sum means looping over rows and accumulating into a dictionary, keyed by group. It works, but you are doing all the split-and-combine bookkeeping by hand.

In [31]:
teams  = ["A", "A", "B", "B", "B"]
scores = [10, 20, 30, 40, 50]

group_sums = {}
for team, score in zip(teams, scores):
    group_sums[team] = group_sums.get(team, 0) + score

print(group_sums)

{'A': 30, 'B': 120}


### 7.2 The same thing with `groupby`

`groupby("team")["score"].sum()` says it in one line: split the rows by `team`, take the `sum` of `score` in each group, and hand back a `Series` indexed by group. The diagram is exactly the numbers below.

![groupby split-apply-combine](assets/pandas/diagram_groupby.png)

In [32]:
games = pd.DataFrame({"team": teams, "score": scores})
games.groupby("team")["score"].sum()

team
A     30
B    120
Name: score, dtype: int64

### 7.3 `groupby` on real data

On Iris, "the average of each feature for each species" is one `groupby`. This is a genuinely useful table: it tells you how the classes differ, feature by feature.

In [33]:
iris.groupby("species")[["petal_length", "petal_width"]].mean()

,petal_length,petal_width
species,,
setosa,1.462,0.246
versicolor,4.260,1.326
virginica,5.552,2.026


### 7.4 Several aggregates at once with `.agg`

`.agg([...])` applies multiple functions per group, and you can name them. Here: the count, mean, and max petal length per species, in one table.

In [34]:
iris.groupby("species")["petal_length"].agg(["count", "mean", "max"])

,count,mean,max
species,,,
setosa,50,1.462,1.9
versicolor,50,4.260,5.1
virginica,50,5.552,6.9


### 7.5 The payoff: `groupby` vs the manual loop, timed

The one-liner is also dramatically faster, because the grouping runs in compiled code instead of a Python loop. Here, a per-key mean gets computed over two million rows both ways and timed. (Both produce the same numbers.)

In [35]:
import numpy as np
import time

rng = np.random.default_rng(0)
n = 2_000_000
big = pd.DataFrame({"key": rng.integers(0, 1000, n), "val": rng.random(n)})

# manual: accumulate sums and counts per key in a Python loop
start = time.perf_counter()
sums, counts = {}, {}
for k, v in zip(big["key"].tolist(), big["val"].tolist()):
    sums[k] = sums.get(k, 0.0) + v
    counts[k] = counts.get(k, 0) + 1
manual = {k: sums[k] / counts[k] for k in sums}
loop_time = time.perf_counter() - start

# pandas
start = time.perf_counter()
fast = big.groupby("key")["val"].mean()
pandas_time = time.perf_counter() - start

# same answer? compare one key
assert abs(manual[0] - fast.loc[0]) < 1e-9

print(f"manual loop: {loop_time:.3f}s")
print(f"groupby:     {pandas_time:.4f}s")
print(f"speedup:     {loop_time / pandas_time:.0f}x")

manual loop: 0.313s
groupby:     0.0208s
speedup:     15x


> **ML tie-in:** per-group aggregates are everywhere: class-conditional feature means, per-user or per-region statistics, and **target encoding** (replacing a category with its group's mean target). `groupby(target).mean()` is also the fastest way to see how your features separate the classes.

## 8. Combining DataFrames: concat and merge

Data rarely lives in one table. You **stack** tables with the same columns (`concat`), and you **join** tables that share a key column (`merge`). These assemble the single dataset your model will train on.

### 8.1 `pd.concat`: stack rows

`pd.concat([...])` glues DataFrames end to end. With the same columns, it stacks rows, the natural way to combine, say, this month's and last month's records.

In [36]:
jan = pd.DataFrame({"name": ["ann", "bob"], "sales": [10, 20]})
feb = pd.DataFrame({"name": ["cara", "dan"], "sales": [30, 40]})
pd.concat([jan, feb], ignore_index=True)

,name,sales
0,ann,10
1,bob,20
2,cara,30
3,dan,40


### 8.2 `pd.merge`: join on a key

`merge` combines two tables by matching a shared **key** column, exactly like a SQL join. Rows are paired up wherever the key agrees. The diagram uses the tables below.

![merge two tables on a key column](assets/pandas/diagram_merge.png)

In [37]:
left  = pd.DataFrame({"id": [1, 2, 3], "name": ["ann", "bob", "cara"]})
right = pd.DataFrame({"id": [1, 2, 4], "dept": ["x", "y", "z"]})
pd.merge(left, right, on="id")

,id,name,dept
0,1,ann,x
1,2,bob,y


### 8.3 `how=`: which rows to keep

By default `merge` is an **inner** join: only keys present in *both* tables survive (above, `id` 3 and 4 dropped out). `how="left"` keeps every row of the left table, filling `NaN` where the right had no match.

In [38]:
pd.merge(left, right, on="id", how="left")

,id,name,dept
0,1,ann,x
1,2,bob,y
2,3,cara,NaN


> **ML tie-in:** feature tables usually come from different sources: a user table, a transactions table, a labels table. `merge` on a shared id is how you assemble them into one modeling dataset, and `concat` is how you append new batches of rows over time.

## 9. Reshaping: pivot and melt

The same data can be laid out **long** (one measurement per row) or **wide** (measurements spread across columns). `pivot` goes long to wide; `melt` goes wide to long. You reshape constantly to match what a plot or a model expects.

### 9.1 `pivot`: long to wide

Start with a long table: one row per (day, item) sale. `pivot` spreads one column's values across the top: pick the new row index, the column to fan out, and the values to fill in. The diagram matches this exact table.

![pivot reshapes long to wide; melt reshapes back](assets/pandas/diagram_pivot_melt.png)

In [39]:
long = pd.DataFrame({
    "day":   ["Mon", "Mon", "Tue", "Tue"],
    "item":  ["A", "B", "A", "B"],
    "sales": [1, 2, 3, 4],
})
wide = long.pivot(index="day", columns="item", values="sales")
wide

item,A,B
day,,
Mon,1,2
Tue,3,4


### 9.2 `melt`: wide back to long

`melt` is the inverse: it collapses a set of columns back into two, a "variable" column of the old names and a "value" column of the entries. Useful when a model or plotting function wants tidy long-format data.

In [40]:
back = wide.reset_index().melt(id_vars="day", var_name="item", value_name="sales")
back

,day,item,sales
0,Mon,A,1
1,Tue,A,3
2,Mon,B,2
3,Tue,B,4


### 9.3 `pivot_table`: pivot with aggregation

Plain `pivot` needs one value per cell. When a cell would collect several rows, `pivot_table` aggregates them (mean by default). It's `groupby` and `pivot` in one step: here, average score per (team, round).

In [41]:
games2 = pd.DataFrame({
    "team":  ["A", "A", "B", "B"],
    "round": [1, 2, 1, 2],
    "score": [10, 20, 30, 40],
})
games2.pivot_table(index="team", columns="round", values="score", aggfunc="mean")

round,1,2
team,,
A,10.0,20.0
B,30.0,40.0


> **ML tie-in:** wide vs long shows up whenever you reshape time series or per-category measurements into a feature matrix, or the other way for a plotting library that expects one row per observation.

## 10. From pandas to the ML stack

pandas is the on-ramp to scikit-learn. Once the table is clean, you hand its numeric core to a model. The bridge is small and worth seeing end to end.

### 10.1 `.to_numpy()`: back to a plain array

A DataFrame's numeric core *is* NumPy underneath. `.to_numpy()` (or `.values`) hands you the raw `(n_rows, n_cols)` array, which is exactly the `(n_samples, n_features)` shape scikit-learn expects.

In [42]:
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X = iris[feature_cols].to_numpy()
print("X type:", type(X).__name__)
print("X shape:", X.shape)   # (n_samples, n_features)

X type: ndarray
X shape: (150, 4)


### 10.2 Building `X` and `y`

The whole point of the cleaning you just did: produce a feature matrix `X` and a target vector `y`. Features are a column selection turned into an array; the target is the label column (as integer codes for the model).

In [43]:
y = iris["species"].astype("category").cat.codes.to_numpy()
print("X shape:", X.shape)
print("y shape:", y.shape)
print("first 5 labels:", y[:5])

X shape: (150, 4)
y shape: (150,)
first 5 labels: [0 0 0 0 0]


### 10.3 The end-to-end handoff

`X` and `y` drop straight into scikit-learn. Here is the real first step of any supervised project: a train/test split, done with the arrays pandas produced. Nothing exotic, just proof that the table you cleaned is now model-ready.

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("train:", X_train.shape, y_train.shape)
print("test: ", X_test.shape, y_test.shape)

train: (120, 4) (120,)
test:  (30, 4) (30,)


> **ML tie-in:** this is the seam between the two libraries. pandas cleans, selects, and engineers; `.to_numpy()` hands off; scikit-learn fits. Every project you build follows this exact path from a raw table to `X`, `y`, and a split.

## 11. Summary

### Cheat sheet

| Task | Tool |
|---|---|
| Create | `pd.Series`, `pd.DataFrame({...})`, `pd.read_csv(...)`, `load_iris(as_frame=True)` |
| Inspect | `.shape`, `.columns`, `.dtypes`, `.head()`/`.tail()`, `.info()`, `.describe()` |
| Select columns | `df["col"]` (Series), `df[["a", "b"]]` (DataFrame) |
| Select by label / position | `.loc[rows, cols]` (labels, end-inclusive), `.iloc[rows, cols]` (positions) |
| Filter rows | boolean mask `df[df["x"] > 5]`, combine with `&` / `|` |
| Add / change columns | `df["new"] = ...`, `.assign(new=...)`, `.map(...)`, `.drop(columns=[...])` |
| Sort / count | `.sort_values(...)`, `.value_counts()`, `.unique()` / `.nunique()` |
| Missing data | `.isna().sum()`, `.dropna()`, `.fillna(...)` |
| Group | `df.groupby("k")["v"].mean()`, `.agg([...])` (split-apply-combine) |
| Combine | `pd.concat([...])` (stack rows), `pd.merge(a, b, on="k", how=...)` (join) |
| Reshape | `.pivot(...)`, `.melt(...)`, `.pivot_table(..., aggfunc=...)` |
| Hand off to ML | `df[cols].to_numpy()` -> `X`, label column -> `y` |

## 12. Exercises

Use only what this notebook covered. Reload Iris the same way as Chapter 1 if you need a fresh copy.

1. Load Iris as a DataFrame and select only the rows where `petal_length` is greater than the mean petal length, keeping just the `species` and `petal_length` columns. How many rows are there?
2. Add a new boolean column `is_big` that is `True` when `petal_area` (`petal_length * petal_width`) is above 10, then use `value_counts` to report how many flowers are "big".
3. Using `groupby`, compute the mean of all four numeric features for each species in a single table. Which species has the largest mean `sepal_length`?
4. The `weight` column of the small table in Chapter 6 has missing values. Fill them with the **median** weight (not the mean), and confirm there are no `NaN` values left.
5. Build `X` (the four feature columns as a NumPy array) and `y` (species as integer codes) from Iris, then use `train_test_split` to hold out 30% for testing. Print the shapes of `X_train` and `X_test`.